# 实验四 · 同步管理与性能度量 —— Event、跨流依赖与设备侧计时

**所属**：《并行计算》第七章 · AscendCL 应用开发　|　**难度**：⭐⭐ 基础　|　**预计时长**：40~50 分钟

把一条多流水线的耗时测清楚，有两件事主机侧墙钟时间做不到：一段耗时里究竟有多少是设备真正在算、多少是下发与同步的开销；以及多条 Stream 同时推进时，哪两段在时间上重叠了。主机只能看到全部完成这一个时刻。

本实验给一条流水线插入 Event 标记。`aclrtEventGetTimestamp` 能取到设备侧的绝对时间戳，把每一块数据的三个阶段的起止时刻都打上标记之后，整条流水线的执行过程就能画成一张时间线——**推断由此变成可以直接读出的事实**。

> **实验说明**
> 1. 本实验的核心内容有三点：Event 的四种用途（跨流依赖、设备侧计时、非阻塞查询、主机等待）、三种计时方式各自测到什么，以及 Event 相对流同步的价值。
> 2. 本实验的程序是一条**多流分块流水线**：数据切成若干块交错下发到多条 Stream 上，并在每一块的每一个阶段前后插入 Event 标记。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 5. 本实验会同时给出**主机侧**与**设备侧**两套时间数据。凡是提到计时方式，指的都是这个数字覆盖了哪一段过程，而不是精度高低。
> 6. 本实验默认读者已经掌握多 Stream 流水线的组织方式：数据分块、异步下发、按 Stream 同步。


## 🎯 学习目标

完成本实验后，学生应能够：

- 说出 Event 的四种用途，并为每一种选出正确的创建 flag
- 指出 `ACL_EVENT_SYNC`、`ACL_EVENT_TIME_LINE` 与 `ACL_EVENT_CAPTURE_STREAM_PROGRESS` 各自对应哪一类用途，以及缺少 `ACL_EVENT_SYNC` 会导致哪个接口不可用
- 正确写出跨流依赖的两步：`aclrtRecordEvent(event, stream)` 与 `aclrtStreamWaitEvent(stream, event)`，并说明两者参数顺序相反
- 说明 Event 相对 `aclrtSynchronizeStream` 的价值：**只阻塞设备侧的目标 Stream，不阻塞主机线程**
- 按四条前提完成一次设备侧计时，并解释为什么必须先同步再取时间
- 区分三种计时方式各自覆盖的过程，并由三者之差算出主机侧开销
- 用 `aclrtEventGetTimestamp` 的绝对时间戳画出多 Stream 的执行时间线，并由它判定两个阶段是否真的重叠
- 用 `aclrtQueryEventStatus` 判断一块内存何时可以安全复用，说明它为什么优于等待整条 Stream


## 🗺️ 学习路径

1. **准备阶段**：认清主机侧墙钟时间的两处不足，它测不出设备的执行时间，也无法观测并发
2. **概念建立**：Event 的四种用途与三个创建 flag，理解同一个对象为什么靠 flag 区分能力
3. **两种时间**：区间长度由 `aclrtEventElapsedTime` 得到，绝对时刻由 `aclrtEventGetTimestamp` 得到，后者才是跨 Stream 比较的依据
4. **程序实现**：为一条多流分块流水线插入 Event 标记，并按三种方式为同一段工作计时
5. **观测重叠**：把设备侧的绝对时刻画成时间线，使两个阶段是否重叠可以由读图得出，不再依赖推断
6. **对照收益**：在同样的工作量下比较 Event 与流同步，说明差别体现在主机被占用的时间上


## 1. 背景与动机：主机侧墙钟时间的两个盲区

到目前为止的测量都用同一种方式：在主机侧记一次时间，做一段事，同步，再记一次时间。这种做法简单可靠，但它有两个盲区。

**盲区一：它测不出一段时间里设备在做什么。** 一条流水线的三段分解通常是这样得到的——下发 H2D、同步、停计时；下发计算、同步、停计时；下发 D2H、同步、停计时。三个数字加起来等于总耗时，看上去很完整。但每一个数字都包含了下发、设备执行、完成通知回传、主机线程被唤醒四段，**主机侧无从知道其中设备真正忙了多久**。本实验要把它拆开。

**盲区二：它看不见并发。** 多条 Stream 同时推进时，主机只能看到全部完成这一个时刻。哪一块数据在什么时候搬、什么时候算，主机侧观测不到。只凭端到端耗时去推断哪两段重叠了，得到的**始终是推断，不是观测**。

Event 补上的正是这两块。它是设备侧的一个标记：插入 Stream 之后随任务一起排队，轮到它时记下当时的时间。由于所有 Event 共用设备上的同一个时钟，不同 Stream 上的 Event 时间戳可以直接比较——**这就得到了一张跨 Stream 的执行时间线。**

### 1.1 本实验要回答的两个问题

1. 主机侧测出的一段耗时里，有多少是设备侧的净执行时间？
2. 多条 Stream 同时推进时，哪两段在时间上真的重叠了？

第一个问题由 §11 的三种计时方式对比回答，第二个问题由 §12 的时间线回答。

### 1.2 本实验不测什么

1. **不做多 Device 之间的同步。** Event 的作用范围是同一个 Device 内的不同 Stream；跨 Device 的通知需要另一套接口，本实验不涉及。
2. **不引入新的性能优化。** 本实验的目的是把这条流水线看清楚，而不是让它更快。


## 2. Event 的概念与创建

### 2.1 一个对象，四种用途

Event 的定义是：**Event 用于同一 Device 内、不同 Stream 之间的任务同步事件，同时支持记录事件时间戳信息。** Event 在实际使用中承担四件事：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 用途 | 关键接口 | 创建 flag | 谁在等 | 本实验相关 |
| --- | --- | --- | --- | --- |
| **跨流依赖** | `aclrtRecordEvent` + `aclrtStreamWaitEvent` | `ACL_EVENT_SYNC` | **设备侧**的另一条 Stream | §13 |
| **设备侧计时** | `aclrtEventElapsedTime` / `aclrtEventGetTimestamp` | `ACL_EVENT_TIME_LINE` | 无人等待，只取时间 | §11、§12 |
| **非阻塞查询** | `aclrtQueryEventStatus` | `ACL_EVENT_CAPTURE_STREAM_PROGRESS` | 无人等待，只问状态 | §5.2 |
| **主机等待** | `aclrtSynchronizeEvent` | `ACL_EVENT_CAPTURE_STREAM_PROGRESS` | **主机线程** | §5.1 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">用途</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">关键接口</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">创建 flag</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">谁在等</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验相关</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>跨流依赖</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtRecordEvent</code> + <code>aclrtStreamWaitEvent</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_SYNC</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>设备侧</strong>的另一条 Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">§13</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>设备侧计时</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtEventElapsedTime</code> / <code>aclrtEventGetTimestamp</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_TIME_LINE</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">无人等待，只取时间</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">§11、§12</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>非阻塞查询</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtQueryEventStatus</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_CAPTURE_STREAM_PROGRESS</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">无人等待，只问状态</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">§5.2</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>主机等待</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSynchronizeEvent</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_CAPTURE_STREAM_PROGRESS</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>主机线程</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">§5.1</td>
</tr>
</tbody>
</table>

四种用途共用同一个对象，但**需要的 flag 不同**，这是本节最容易出错的地方。

### 2.2 创建：flag 决定这个 Event 能做什么

```cpp
aclError aclrtCreateEventExWithFlag(aclrtEvent *event, uint32_t flag);
aclError aclrtDestroyEvent(aclrtEvent event);
```

`flag` 指定这个 Event 的用途。三个宏的对应场景如下：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 宏 | 使能的功能 | 对应的场景 |
| --- | --- | --- |
| `ACL_EVENT_SYNC` | 多 Stream 之间的同步 | Event 等待：`aclrtStreamWaitEvent` |
| `ACL_EVENT_TIME_LINE` | 记录事件时间戳 | 记录 Event 时间戳：`aclrtEventElapsedTime` |
| `ACL_EVENT_CAPTURE_STREAM_PROGRESS` | 捕获 Stream 的执行进度 | Event 查询：`aclrtQueryEventStatus`；Event 同步：`aclrtSynchronizeEvent` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">宏</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">使能的功能</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对应的场景</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_SYNC</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多 Stream 之间的同步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Event 等待：<code>aclrtStreamWaitEvent</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_TIME_LINE</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">记录事件时间戳</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">记录 Event 时间戳：<code>aclrtEventElapsedTime</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_CAPTURE_STREAM_PROGRESS</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">捕获 Stream 的执行进度</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Event 查询：<code>aclrtQueryEventStatus</code>；Event 同步：<code>aclrtSynchronizeEvent</code></td>
</tr>
</tbody>
</table>

由此得到一条选用规则：

- 只做跨流依赖 → `ACL_EVENT_SYNC`
- 只做计时 → `ACL_EVENT_TIME_LINE`
- 只被主机查询或等待 → `ACL_EVENT_CAPTURE_STREAM_PROGRESS`

本实验 §12 的 `timeline` 版本要跑一个既被记录时间戳、又带同步语义的 Event，传的是 `ACL_EVENT_TIME_LINE | ACL_EVENT_SYNC`；

还有一条与测量直接相关的说法：**使能时间戳功能会影响 Event 相关接口的性能**。它的含义是**计时这件事本身有代价**——用来观察流水线的标记会改变流水线的行为。这不是接口的缺陷，而是一切侵入式测量的共同性质。

### 2.3 数量限制与一条容易忽略的时序

单个 Device 支持的 Event 数量是有上限的，`aclrtGetEventAvailNum(uint32_t *eventCount)` 查询当前 Device 上剩余可用的数量。**具体上限以本机的实测值为准**：§6 的探针会把它打印出来。

资源是**什么时候**申请的同样值得注意：创建 Event 之后，要到调用 `aclrtRecordEvent` 时系统内部才会真正申请 Event 资源，因此数量限制是在 Record 时才生效的。创建了大量 Event 却不 Record，不会触及上限；反过来，Record 到达上限时系统内部会等待资源释放，表现为一次意料之外的停顿。

本实验的时间线模式为每一块数据创建 4 个 Event，分块数取 8 时共 32 个，远在限制之内。

### 2.4 Record 捕获的是什么

```cpp
aclError aclrtRecordEvent(aclrtEvent event, aclrtStream stream);
```

这个接口的语义：**调用时会捕获当前 Stream 上已下发的任务，并记录到 Event 中。** 包含三层含义：

1. **它是异步接口**：调用返回不代表被捕获的任务已完成，只表示这个记录动作被排进了队列。
2. **它捕获的是此刻为止已下发的全部任务**，而不是某一个任务。因此 Record 的位置决定了 Event 的含义——插在 H2D 之后，它就代表 H2D 完成。
3. **同一个 Event 可以多次 Record**，每次重新捕获并覆盖。这使得在循环里复用少量 Event 成为可能，但也意味着**一个 Event 在同一时刻只代表最近一次 Record 的那批任务**。本实验为每个阶段准备独立的 Event，不复用，以免时间戳互相覆盖。


## 3. 三种计时方式

同一段 H2D + 计算 + D2H，用三种方法计时会得到三个差别很大的数字。它们不是精度不同，而是**覆盖的过程不同**。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 计时方式 | 做法 | 覆盖的过程 | 典型量级 |
| --- | --- | --- | --- |
| **主机侧 · 不同步** | 记时间 → 下发 → 记时间 | **只有下发**：接口调用、参数校验、任务入队 | 极小 |
| **主机侧 · 同步后** | 记时间 → 下发 → 同步 → 记时间 | 端到端：下发 + 设备执行 + 完成通知 + 主机唤醒 | 最大 |
| **设备侧 · Event** | Record → 任务 → Record → 同步 → 取差值 | **设备上任务的净执行区间** | 居中 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">计时方式</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">做法</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">覆盖的过程</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">典型量级</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>主机侧 · 不同步</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">记时间 → 下发 → 记时间</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>只有下发</strong>：接口调用、参数校验、任务入队</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">极小</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>主机侧 · 同步后</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">记时间 → 下发 → 同步 → 记时间</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">端到端：下发 + 设备执行 + 完成通知 + 主机唤醒</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">最大</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>设备侧 · Event</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Record → 任务 → Record → 同步 → 取差值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>设备上任务的净执行区间</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">居中</td>
</tr>
</tbody>
</table>

三者之间的差值各有含义：

$$\underbrace{T_{\text{同步后}} - T_{\text{设备侧}}}_{\text{主机侧开销}}
\qquad
\underbrace{T_{\text{设备侧}}}_{\text{设备真正在忙}}
\qquad
\underbrace{T_{\text{不同步}}}_{\text{下发本身}}$$

三个数字里，**不同步这一种**通常只有另外两个的百分之一量级；而**主机侧开销**（同步后减设备侧）是一个与数据量无关的固定量，它在端到端中占多大比例，取决于本机的设备侧耗时有多长——设备侧越短，这个固定量越显眼。两者都请在 §11 用本机的输出读出来。第六章反复强调过核函数启动是异步的、不同步测到的只是下发耗时，那时只能定性地说；本实验把它变成一个可以读出的比例。

### 3.1 设备侧计时的四条前提

```cpp
aclError aclrtEventElapsedTime(float *ms, aclrtEvent startEvent,
                               aclrtEvent endEvent);
```

1. **Event 必须带 `ACL_EVENT_TIME_LINE` 创建。** 不带这个 flag 的 Event 没有时间戳可取。
2. **起止两个 Event 与被测任务在同一条 Stream 上。** 跨 Stream 的两个 Event 之差不表示一段任务的耗时，跨 Stream 的时间比较应当使用 §12 的绝对时间戳。
3. **必须先 `aclrtSynchronizeStream` 再取时间。** Record 是异步的，不同步就去读，读到的是尚未写入的值。调用顺序很明确：创建 → Record 起止 → **同步** → 取值。
4. **输出是 `float`，以毫秒为单位。** 注意单精度的有效位有限，两个相距很远的时间点相减会损失精度；需要绝对时刻时应当用 `aclrtEventGetTimestamp`。

### 3.2 绝对时间戳：画时间线的前提

```cpp
aclError aclrtEventGetTimestamp(aclrtEvent event, uint64_t *timestamp);
```

它返回的是**该 Event 的执行结束时间点，单位为微秒**，基准是昇腾 AI 处理器系统启动以来的时间。§10 会用 `/1000.0` 把它换算成毫秒，换算后与主机侧的 `total_ms` 对得上。两点值得注意：

- **它是绝对时刻，不是区间。** 因此可以把不同 Stream 上的 Event 放在同一根时间轴上比较——这是 §12 能画出甘特图的依据，`aclrtEventElapsedTime` 做不到这一点，因为它只给出成对的差值。
- **它是 `uint64_t` 而不是 `float`。** 与第三条前提配合：需要跨 Stream 比较绝对时刻时用它，只需要一段区间的长度时用 `aclrtEventElapsedTime` 更直接。

调用顺序与 `aclrtEventElapsedTime` 相同，**同样必须先同步**。


## 4. 跨流依赖：Record 与 StreamWaitEvent

### 4.1 两步构造

不同 Stream 之间不存在任何隐式的先后关系。要表达 stream2 的任务必须等 stream1 的任务完成，需要两步：

```cpp
aclrtRecordEvent(event, stream1);      // 在 stream1 中插入 Event Record 任务
aclrtStreamWaitEvent(stream2, event);  // 在 stream2 中插入 Event Wait 任务
```

<img src="images/07.04_event_wait.png" alt="一个 Stream 的任务等待另一个 Stream 的 Event" width="579px">

图中 stream1 执行完 task1 后触发 Event Record，stream2 中的 Event Wait 任务在此之前一直挡住 task2；Event 完成后 task2 才开始。**两个 Stream 的任务队列各自独立推进，唯一的连接点是这个 Event。**

> ⚠️ **两个接口的参数顺序是相反的**：`aclrtRecordEvent(event, stream)`，而 `aclrtStreamWaitEvent(stream, event)`。这是本节最高频的错误，且**写反了未必立刻报错**——两个参数都是指针类型，编译器不会拦住。写的时候可以记一句话：Record 是把 Event 记进某条流，Wait 是让某条流去等 Event，**主语不同，所以第一个参数不同**。

Event 也支持**多等一**：多条 Stream 等待同一个 Event。

<img src="images/07.04_event_multi_wait.png" alt="多条 Stream 等待同一个 Event" width="590px">

反过来的一等多需要多个 Event、多次 `aclrtStreamWaitEvent`，接口本身不提供聚合语义。

### 4.2 它相对流同步的价值

同样要表达依赖，也可以在主机侧调用 `aclrtSynchronizeStream(stream1)`，等它完成之后再向 stream2 下发。两种写法的结果都正确，差别在于**谁在等**：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 写法 | 阻塞对象 | 此期间主机能否继续下发 | 适用场合 |
| --- | --- | --- | --- |
| `aclrtSynchronizeStream(stream1)` | **主机线程** | 不能——主机被挡住了 | 确实需要取回结果到主机时 |
| Record + `aclrtStreamWaitEvent` | **设备侧的 stream2** | **能**——主机不受影响 | 依赖关系发生在设备内部时 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">写法</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">阻塞对象</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">此期间主机能否继续下发</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">适用场合</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSynchronizeStream(stream1)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>主机线程</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不能——主机被挡住了</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">确实需要取回结果到主机时</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Record + <code>aclrtStreamWaitEvent</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>设备侧的 stream2</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>能</strong>——主机不受影响</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">依赖关系发生在设备内部时</td>
</tr>
</tbody>
</table>

差别在什么时候显现，取决于**有没有不依赖 stream1 的任务可以先下发**。若有，Event 写法可以把它们提前排进 stream2，让它们在等待期间就执行；流同步写法则会把主机挡在原地，这些任务连下发的机会都没有。

**§13 用一组对照测量把这个差别量化出来**：两个版本执行完全相同的工作量，唯一的差别是依赖的表达方式，比较总耗时与主机被阻塞的时间。

### 4.3 一条时序上的要求

`aclrtStreamWaitEvent` 等待的是**该 Event 最近一次 Record 所捕获的任务**。因此两个调用的**下发顺序**是有意义的：必须先 Record 再 Wait。若顺序颠倒，Wait 等到的是上一次 Record 的内容（或者在从未 Record 时立即通过），依赖关系就失效了——**而且程序不会报错，只会偶发地读到未完成的数据**。


## 5. 非阻塞查询与主机等待

### 5.1 三种等待的粒度

Event 一侧也有三种粒度，与 Stream 一侧的三种同步接口一一对应：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 接口 | 谁在等 | 是否阻塞主机 | 典型用途 |
| --- | --- | --- | --- |
| `aclrtStreamWaitEvent(stream, event)` | 设备侧的指定 Stream | **否** | 表达跨流依赖 |
| `aclrtSynchronizeEvent(event)` | **主机线程** | 是 | 只等一段任务，不等整条 Stream |
| `aclrtQueryEventStatus(event, &status)` | 无人等待 | **否** | 轮询进度、判断资源可否复用 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">接口</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">谁在等</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">是否阻塞主机</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">典型用途</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtStreamWaitEvent(stream, event)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">设备侧的指定 Stream</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>否</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">表达跨流依赖</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtSynchronizeEvent(event)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>主机线程</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">是</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">只等一段任务，不等整条 Stream</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtQueryEventStatus(event, &amp;status)</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">无人等待</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>否</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">轮询进度、判断资源可否复用</td>
</tr>
</tbody>
</table>

`aclrtSynchronizeEvent` 值得单独一提：它比 `aclrtSynchronizeStream` 精细——**只等 Event 捕获的那批任务，而不是整条 Stream 上的全部任务**。当一条 Stream 上排了很多任务而主机只关心前面一段时，用它可以早得多地返回。

### 5.2 非阻塞查询与内存复用判定

```cpp
ACL_EVENT_RECORDED_STATUS_NOT_READY   // 有未执行完的任务
ACL_EVENT_RECORDED_STATUS_COMPLETE    // 所有任务都已经执行完成
```

这个接口回答了一个实际问题：异步传输期间源内存与目的内存都不能改动，那么何时才可以？

答案是：**在需要复用的那块内存最后一次被用到之后 Record 一个 Event，当它变为 `COMPLETE` 时即可安全复用。** 关键在于这个时刻通常**远早于整条 Stream 完成**——一条 Stream 上可能还排着几十块数据的任务，而第一块的输入缓冲区在它自己的 H2D 结束时就已经自由了。按各自的 Event 释放缓冲区，主机侧只需准备很少几块就能维持流水线满载——这正是内存池的原理。

## 6. 环境准备与检查

先把 CANN 的环境变量导入 Jupyter 进程，并创建代码目录。


In [ ]:
!mkdir -p src_event

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")


本实验需要三个库：Runtime 库、算子公共数据类型库与 Math 类算子库。


In [ ]:
import os, shutil

ascend_home = os.environ.get("ASCEND_HOME_PATH", "")
lib_dirs = [
    path
    for path in (f"{ascend_home}/lib64", f"{ascend_home}/devlib")
    if os.path.isdir(path)
]


# 按优先级在库目录中查找库，返回第一个存在的库名（不含 lib 前缀与 .so 后缀）
def find_lib(candidates):
    for name in candidates:
        for lib_dir in lib_dirs:
            if os.path.exists(os.path.join(lib_dir, f"lib{name}.so")):
                return name
    return None


print("ASCEND_HOME_PATH :", ascend_home or "⚠️  未设置")
print("g++              :", shutil.which("g++") or "⚠️  未找到")

selected = []
for purpose, candidates in [
    ("Runtime", ["acl_rt", "ascendcl"]),
    ("算子公共数据类型", ["nnopbase"]),
    ("Math 类算子", ["opapi_math"]),
]:
    name = find_lib(candidates)
    print(f"{purpose:<18} 候选 {candidates} -> {name}")
    if name is not None:
        selected.append(name)

ACL_LIBDIRS = ["-L" + path for path in lib_dirs]
ACL_LIBS = ["-l" + name for name in selected]
ACL_RT_LIB = ["-l" + selected[0]] if selected else []
print("链接参数         :", " ".join(ACL_LIBS))
print()
print(
    "✅ 环境就绪，可以开始实验。"
    if ascend_home and shutil.which("g++") and len(selected) == 3
    else "⚠️  环境不完整，请检查上面的输出。"
)


下面这个探针查询当前 Device 上剩余可用的 Event 数量。§2.3 提到过，Event 资源是在 `aclrtRecordEvent` 时才真正申请的，因此这个数字在程序运行前后可能不同；这里查询的是运行前的起点。

In [ ]:
%%writefile src_event/event_probe.cpp
// Prints how many events are still available on the device. Event resources
// are allocated when aclrtRecordEvent is called rather than at creation time,
// so this is the count before any of this lab's events are recorded.
// aclrtGetEventAvailNum takes exactly one argument: the device is the one
// selected by the preceding aclrtSetDevice call, not a parameter of its own.
#include <cstdint>
#include <cstdio>

#include "acl/acl.h"

int main() {
  if (aclInit(nullptr) != ACL_SUCCESS || aclrtSetDevice(0) != ACL_SUCCESS) {
    std::printf("[ERR] failed to initialise the device\n");
    return 1;
  }

  uint32_t available = 0;
  if (aclrtGetEventAvailNum(&available) == ACL_SUCCESS) {
    std::printf("[PROBE] events_available=%u\n", available);
  } else {
    std::printf("[PROBE] events_available=unknown\n");
  }

  aclrtResetDevice(0);
  aclFinalize();
  return 0;
}


编译并运行探针。它只包含 `acl/acl.h`，因此只链接 Runtime 库。


In [ ]:
import os, subprocess

cmd = (
    ["g++", "src_event/event_probe.cpp", "-std=c++17", "-O2", "-Wall"]
    + ["-I" + os.environ["ASCEND_HOME_PATH"] + "/include"]
    + ACL_LIBDIRS
    + ACL_RT_LIB
    + ["-o", "src_event/event_probe"]
)
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print((proc.stdout + proc.stderr).strip())
    print()
    print("❌ 探针编译失败（返回码 %d）：Event 可用数未取到。" % proc.returncode)
    print("   请对照 §2.3 检查 aclrtGetEventAvailNum 的参数——它只接收一个")
    print("   uint32_t* 出参，Device 由此前的 aclrtSetDevice 决定。")
    print("   修好后重新执行本节两格；不修也可以继续，后文不使用这个数字。")
else:
    run = subprocess.run(
        ["./src_event/event_probe"], capture_output=True, text=True
    )
    print(run.stdout.strip())
    print("✅ 探针运行完成" if run.returncode == 0
          else "❌ 探针运行失败（返回码 %d）" % run.returncode)


## 7. 本实验的测量设计

三组测量各回答一个问题，共用同一条流水线。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 模式 | 问题 | 手段 | 落点 |
| --- | --- | --- | --- |
| `timing` | 三种计时方式各测到什么？主机侧开销有多大？ | 同一段任务三种计时并排 | §11 |
| `timeline` | **多条 Stream 上哪两段在时间上重叠？** | 每块每阶段前后各打一个绝对时间戳 | §12 |
| `depend` | Event 相对流同步值多少？ | 同样的工作量，两种依赖表达方式对照 | §13 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">模式</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">问题</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">手段</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">落点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>timing</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三种计时方式各测到什么？主机侧开销有多大？</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同一段任务三种计时并排</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">§11</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>timeline</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>多条 Stream 上哪两段在时间上重叠？</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每块每阶段前后各打一个绝对时间戳</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">§12</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>depend</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Event 相对流同步值多少？</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同样的工作量，两种依赖表达方式对照</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">§13</td>
</tr>
</tbody>
</table>

### 7.1 参数的选择

总量取 64 MB，**分块数取 8、流数取 2**，计算重复次数取 4。

分块数与流数都取得较小，理由有两条：其一，§12 的时间线要逐块画出三个阶段，块数太多会互相遮挡，8 块 2 流每条 Stream 恰好 4 块；其二，两条 Stream 已经足以呈现跨流比较这件事，本实验要看的是重叠关系而不是最大吞吐。读 §12 的图时请记住：**这里画的不是本机吞吐最高的那个配置**，本实验要回答的问题与绝对值无关。

### 7.2 测量方法

下面六条要点在本实验中全部沿用：首次触碰、预热、多次重复取平均、分别记录下发与总耗时、全部 Stream 都要同步、张量与内存在计时之外准备。本实验新增一条：

**取时间戳之前必须先同步。** 所有 `aclrtEventGetTimestamp` 与 `aclrtEventElapsedTime` 都放在 `aclrtSynchronizeStream` 之后。Record 是异步下发的，不同步就去读，读到的是尚未写入的值。


## 8. 程序实现

程序分六段写入 `src_event/acl_event.cpp`。第一段为覆盖写，其余五段为追加写，因此**必须按顺序执行**。

### 8.1 头文件、常量与错误检查

程序接受五个命令行参数：模式、总数据量（MB）、分块数、流数、计算重复次数。


In [ ]:
%%writefile src_event/acl_event.cpp
/**
 * Parallel Computing, Chapter 7, Lab 4: Synchronization and Measurement
 *
 * This program instruments a multi-stream chunked pipeline with events. It
 * answers two questions that host-side wall clock timing cannot: how much of a
 * measured
 * interval is spent executing on the device, and whether two stages that run
 * on different streams actually overlap in time.
 *
 * Usage: acl_event <mode> <total_mb> <chunks> <streams> <repeat>
 *   mode = timing | timeline | depend
 */
#include <cstdint>  // int32_t, int64_t, uint64_t
#include <cstdio>   // std::printf, std::fprintf
#include <cstdlib>  // std::atoi
#include <cstring>  // std::strcmp
#include <ctime>    // clock_gettime, timespec

#include "acl/acl.h"            // Runtime resource management APIs
#include "aclnnop/aclnn_add.h"  // Single-operator API of Add

namespace {

constexpr int32_t kDeviceId = 0;
constexpr int kMaxStreams = 8;
constexpr int kMaxChunks = 64;
constexpr int kEventsPerChunk = 4;
constexpr int kWarmupRuns = 2;
constexpr int kMeasureRuns = 5;
constexpr float kAlphaValue = 1.0f;
constexpr float kBiasValue = 1.0f;
constexpr double kRelTolerance = 1.0e-6;
// An event that is only used for timing needs ACL_EVENT_TIME_LINE; one that is
// also used by aclrtStreamWaitEvent must additionally carry ACL_EVENT_SYNC.
constexpr uint32_t kTimingFlag = ACL_EVENT_TIME_LINE;
constexpr uint32_t kSyncFlag = ACL_EVENT_SYNC;

}  // namespace

// Checks the return code of an acl API. On failure it prints the API name,
// the return code and the error message, then returns immediately.
#define ACL_CHECK(expr)                                                     \
  do {                                                                      \
    const int acl_ret = static_cast<int>(expr);                             \
    if (acl_ret != ACL_SUCCESS) {                                           \
      const char* err_msg = aclGetRecentErrMsg();                           \
      std::fprintf(stderr, "[ERR] api=%s code=%d msg=%s\n", #expr, acl_ret, \
                   (err_msg == nullptr) ? "(no message)" : err_msg);        \
      return acl_ret;                                                       \
    }                                                                       \
  } while (0)


### 8.2 计时工具与算子封装

`GetTimeMs` 取主机侧单调时钟，`SubmitAdd` 把算子的两段式调用包成一次提交。


In [ ]:
%%writefile -a src_event/acl_event.cpp

// Returns a monotonic timestamp in milliseconds, for the host-side clock.
double GetTimeMs() {
  timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1.0e6;
}

// Submits one Add task to a stream. The first phase runs on the host and the
// second phase submits the kernel.
int SubmitAdd(const aclTensor* self, const aclTensor* other,
              const aclScalar* alpha, aclTensor* out, void* workspace,
              uint64_t workspace_capacity, aclrtStream stream) {
  uint64_t needed = 0;
  aclOpExecutor* executor = nullptr;
  ACL_CHECK(
      aclnnAddGetWorkspaceSize(self, other, alpha, out, &needed, &executor));
  if (needed > workspace_capacity) {
    std::fprintf(stderr,
                 "[ERR] api=SubmitAdd code=- msg=workspace too small\n");
    return ACL_ERROR_INVALID_PARAM;
  }
  ACL_CHECK(aclnnAdd(workspace, needed, executor, stream));
  return ACL_SUCCESS;
}


### 8.3 资源集合

`Pipeline` 持有流水线的全部资源，其中与本实验直接相关的是一组 Event。`events[c][j]` 是第 $c$ 块的第 $j$ 个标记点，四个标记点依次表示：本块开始、H2D 完成、计算完成、D2H 完成。

`use_events` 是 §7.2 提到的那个开关：置为假时全部 Record 调用都被跳过，用于跑出一份不带标记的基准。


In [ ]:
%%writefile -a src_event/acl_event.cpp

// Holds every resource the instrumented pipeline needs. The host buffers are
// page-locked because aclrtMemcpyAsync is only truly asynchronous on them.
struct Pipeline {
  int64_t total_elems;
  int64_t chunk_elems;
  int chunks;
  // How many streams were actually created, always kMaxStreams: the pipeline
  // allocates the full set once and each run uses only the first few. The
  // number of streams a run uses is passed separately, it is not this field.
  int streams;
  bool use_events;
  float* host_in;
  float* host_out;
  void* dev_in;
  void* dev_out;
  void* dev_bias;
  aclrtStream stream[kMaxStreams];
  void* workspace[kMaxStreams];
  uint64_t workspace_size;
  aclTensor* in_tensor[kMaxChunks];
  aclTensor* out_tensor[kMaxChunks];
  aclTensor* bias_tensor;
  aclScalar* alpha;
  aclrtEvent event[kMaxChunks][kEventsPerChunk];
  int64_t shape[1];
  int64_t stride[1];
};

// Records a marker on a stream unless the instrumentation is switched off.
int MarkPoint(Pipeline* p, int chunk, int slot, aclrtStream stream) {
  if (!p->use_events) {
    return ACL_SUCCESS;
  }
  ACL_CHECK(aclrtRecordEvent(p->event[chunk][slot], stream));
  return ACL_SUCCESS;
}

// Creates one event per marker point. The flag decides what the event can be
// used for: timing needs ACL_EVENT_TIME_LINE, cross-stream waiting needs
// ACL_EVENT_SYNC, querying or host-side waiting needs
// ACL_EVENT_CAPTURE_STREAM_PROGRESS, and an event used both for timing and for
// waiting must carry the first two of them.
int CreateMarkers(Pipeline* p, uint32_t flag) {
  for (int c = 0; c < p->chunks; ++c) {
    for (int j = 0; j < kEventsPerChunk; ++j) {
      ACL_CHECK(aclrtCreateEventExWithFlag(&p->event[c][j], flag));
    }
  }
  return ACL_SUCCESS;
}

int DestroyMarkers(Pipeline* p) {
  for (int c = 0; c < p->chunks; ++c) {
    for (int j = 0; j < kEventsPerChunk; ++j) {
      if (p->event[c][j] != nullptr) {
        ACL_CHECK(aclrtDestroyEvent(p->event[c][j]));
        p->event[c][j] = nullptr;
      }
    }
  }
  return ACL_SUCCESS;
}


分配、释放与校验的写法是常规的，此处只做必要的说明：主机缓冲区用 `aclrtMallocHost` 申请（锁页内存是异步传输真正异步的前提），`bias` 常驻设备侧不参与每块的传输，首次触碰在计时之外完成。


In [ ]:
%%writefile -a src_event/acl_event.cpp

namespace {
// Upper bound reserved for the operator workspace of one stream.
constexpr uint64_t kWorkspaceReserve = 16ULL << 20;
}  // namespace

aclTensor* MakeTensor(Pipeline* p, void* addr) {
  return aclCreateTensor(p->shape, 1, ACL_FLOAT, p->stride, 0, ACL_FORMAT_ND,
                         p->shape, 1, addr);
}

int AllocPipeline(Pipeline* p, int64_t total_elems, int chunks, int streams) {
  p->total_elems = total_elems;
  p->chunks = chunks;
  p->streams = streams;
  p->chunk_elems = total_elems / chunks;
  p->use_events = true;
  p->workspace_size = kWorkspaceReserve;
  p->shape[0] = p->chunk_elems;
  p->stride[0] = 1;
  const size_t bytes = static_cast<size_t>(total_elems) * sizeof(float);

  ACL_CHECK(aclrtMallocHost(reinterpret_cast<void**>(&p->host_in), bytes));
  ACL_CHECK(aclrtMallocHost(reinterpret_cast<void**>(&p->host_out), bytes));
  ACL_CHECK(aclrtMalloc(&p->dev_in, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc(&p->dev_out, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc(&p->dev_bias, bytes, ACL_MEM_MALLOC_HUGE_FIRST));

  // First touch, outside every timed region.
  for (int64_t i = 0; i < total_elems; ++i) {
    p->host_in[i] = static_cast<float>(i % 1024) * 0.25f;
    p->host_out[i] = kBiasValue;
  }
  ACL_CHECK(aclrtMemcpy(p->dev_bias, bytes, p->host_out, bytes,
                        ACL_MEMCPY_HOST_TO_DEVICE));
  for (int64_t i = 0; i < total_elems; ++i) {
    p->host_out[i] = 0.0f;
  }

  for (int s = 0; s < streams; ++s) {
    ACL_CHECK(aclrtCreateStream(&p->stream[s]));
    ACL_CHECK(aclrtMalloc(&p->workspace[s], kWorkspaceReserve,
                          ACL_MEM_MALLOC_HUGE_FIRST));
  }
  float alpha_value = kAlphaValue;
  p->alpha = aclCreateScalar(&alpha_value, ACL_FLOAT);
  for (int c = 0; c < chunks; ++c) {
    const int64_t offset = static_cast<int64_t>(c) * p->chunk_elems;
    p->in_tensor[c] = MakeTensor(p, static_cast<float*>(p->dev_in) + offset);
    p->out_tensor[c] = MakeTensor(p, static_cast<float*>(p->dev_out) + offset);
  }
  p->bias_tensor = MakeTensor(p, p->dev_bias);
  return ACL_SUCCESS;
}

int FreePipeline(Pipeline* p) {
  ACL_CHECK(DestroyMarkers(p));
  for (int c = 0; c < p->chunks; ++c) {
    ACL_CHECK(aclDestroyTensor(p->in_tensor[c]));
    ACL_CHECK(aclDestroyTensor(p->out_tensor[c]));
  }
  ACL_CHECK(aclDestroyTensor(p->bias_tensor));
  ACL_CHECK(aclDestroyScalar(p->alpha));
  for (int s = 0; s < p->streams; ++s) {
    ACL_CHECK(aclrtFree(p->workspace[s]));
    ACL_CHECK(aclrtDestroyStream(p->stream[s]));
  }
  ACL_CHECK(aclrtFree(p->dev_bias));
  ACL_CHECK(aclrtFree(p->dev_out));
  ACL_CHECK(aclrtFree(p->dev_in));
  ACL_CHECK(aclrtFreeHost(p->host_out));
  ACL_CHECK(aclrtFreeHost(p->host_in));
  return ACL_SUCCESS;
}

// The expected result is out = in + alpha * bias regardless of how many times
// the computation was repeated. This is arithmetic, so a relative-error
// tolerance is the right tool here, unlike the byte comparison of Lab 2.
int VerifyResult(const Pipeline* p, const char* tag) {
  double max_rel = 0.0;
  for (int64_t i = 0; i < p->total_elems; ++i) {
    const double want =
        static_cast<double>(p->host_in[i]) +
        static_cast<double>(kAlphaValue) * static_cast<double>(kBiasValue);
    const double got = static_cast<double>(p->host_out[i]);
    const double denom = (want == 0.0) ? 1.0 : (want < 0.0 ? -want : want);
    const double err = (got - want < 0.0) ? (want - got) : (got - want);
    if (err / denom > max_rel) {
      max_rel = err / denom;
    }
  }
  const bool ok = max_rel <= kRelTolerance;
  std::printf("[VERIFY] tag=%s max_rel_err=%.3e result=%s\n", tag, max_rel,
              ok ? "PASS" : "FAIL");
  return ok ? ACL_SUCCESS : ACL_ERROR_INVALID_PARAM;
}


### 8.4 流水线主体与两种测量

`RunPipeline` 是流水线主体：在每一块的三个阶段前后调用 `MarkPoint` 插入标记；开关关闭时它退化成不带标记的原版。

`RunThreeClocks` 实现 §11 的三种计时方式。三个时间点的取法值得逐行看：`t1` 在下发循环结束时取，此时设备可能一个任务都没做完；`t2` 在同步之后取；设备侧的时间则由界定全部任务的两个 Event 相减得到。**三者测的是同一段工作，差别只在覆盖范围。**


In [ ]:
%%writefile -a src_event/acl_event.cpp

// A multi-stream chunked pipeline with four marker points per chunk.
// With use_events switched off every marker call is skipped, so the same
// function also runs the pipeline without any instrumentation.
int RunPipeline(Pipeline* p, int streams, int repeat, double* submit_ms,
                double* total_ms) {
  const size_t chunk_bytes =
      static_cast<size_t>(p->chunk_elems) * sizeof(float);
  const double t0 = GetTimeMs();
  for (int c = 0; c < p->chunks; ++c) {
    const int slot = c % streams;
    aclrtStream stream = p->stream[slot];
    const int64_t offset = static_cast<int64_t>(c) * p->chunk_elems;
    ACL_CHECK(MarkPoint(p, c, 0, stream));
    ACL_CHECK(aclrtMemcpyAsync(static_cast<float*>(p->dev_in) + offset,
                               chunk_bytes, p->host_in + offset, chunk_bytes,
                               ACL_MEMCPY_HOST_TO_DEVICE, stream));
    ACL_CHECK(MarkPoint(p, c, 1, stream));
    for (int r = 0; r < repeat; ++r) {
      ACL_CHECK(SubmitAdd(p->in_tensor[c], p->bias_tensor, p->alpha,
                          p->out_tensor[c], p->workspace[slot],
                          p->workspace_size, stream));
    }
    ACL_CHECK(MarkPoint(p, c, 2, stream));
    ACL_CHECK(aclrtMemcpyAsync(p->host_out + offset, chunk_bytes,
                               static_cast<float*>(p->dev_out) + offset,
                               chunk_bytes, ACL_MEMCPY_DEVICE_TO_HOST, stream));
    ACL_CHECK(MarkPoint(p, c, 3, stream));
  }
  const double t1 = GetTimeMs();
  for (int s = 0; s < streams; ++s) {
    ACL_CHECK(aclrtSynchronizeStream(p->stream[s]));
  }
  *submit_ms = t1 - t0;
  *total_ms = GetTimeMs() - t0;
  return ACL_SUCCESS;
}

// Times one piece of work three ways at once. All three numbers describe the
// same work; they differ only in which part of the process they cover.
int RunThreeClocks(Pipeline* p, int repeat, int runs) {
  const size_t bytes = static_cast<size_t>(p->total_elems) * sizeof(float);
  aclrtStream stream = p->stream[0];
  aclrtEvent begin = nullptr;
  aclrtEvent end = nullptr;
  ACL_CHECK(aclrtCreateEventExWithFlag(&begin, kTimingFlag));
  ACL_CHECK(aclrtCreateEventExWithFlag(&end, kTimingFlag));

  double submit_sum = 0.0;
  double wall_sum = 0.0;
  double device_sum = 0.0;
  for (int i = 0; i < kWarmupRuns + runs; ++i) {
    const double t0 = GetTimeMs();
    ACL_CHECK(aclrtRecordEvent(begin, stream));
    ACL_CHECK(aclrtMemcpyAsync(p->dev_in, bytes, p->host_in, bytes,
                               ACL_MEMCPY_HOST_TO_DEVICE, stream));
    for (int r = 0; r < repeat; ++r) {
      ACL_CHECK(SubmitAdd(p->in_tensor[0], p->bias_tensor, p->alpha,
                          p->out_tensor[0], p->workspace[0], p->workspace_size,
                          stream));
    }
    ACL_CHECK(aclrtMemcpyAsync(p->host_out, bytes, p->dev_out, bytes,
                               ACL_MEMCPY_DEVICE_TO_HOST, stream));
    ACL_CHECK(aclrtRecordEvent(end, stream));
    const double t1 = GetTimeMs();
    // The timestamps are only valid once the stream has been synchronized.
    ACL_CHECK(aclrtSynchronizeStream(stream));
    const double t2 = GetTimeMs();
    float device_ms = 0.0f;
    ACL_CHECK(aclrtEventElapsedTime(&device_ms, begin, end));
    if (i >= kWarmupRuns) {
      submit_sum += t1 - t0;
      wall_sum += t2 - t0;
      device_sum += static_cast<double>(device_ms);
    }
  }
  std::printf(
      "[CLOCK] submit_ms=%.4f wall_ms=%.4f device_ms=%.4f overhead_ms=%.4f\n",
      submit_sum / runs, wall_sum / runs, device_sum / runs,
      (wall_sum - device_sum) / runs);

  ACL_CHECK(aclrtDestroyEvent(end));
  ACL_CHECK(aclrtDestroyEvent(begin));
  return ACL_SUCCESS;
}


### 8.5 时间线与跨流依赖

`RunTimeline` 把每一块四个标记点的**绝对时间戳**打印出来。由于全部 Event 共用设备上的同一个时钟，不同 Stream 上的时间戳可以直接比较——这是画甘特图的依据。

`RunDependency` 是 §13 的对照：两个版本做完全相同的工作，唯一差别是依赖的表达方式。**Event 版本刻意把不依赖 stream1 的那部分任务放在 `aclrtStreamWaitEvent` 之前下发**，这样它们不必等待就能开始执行；流同步版本则做不到，因为主机线程被挡在了同步调用里。


In [ ]:
%%writefile -a src_event/acl_event.cpp

// Prints the absolute device-side timestamp of every marker point. All events
// share one device clock, so timestamps from different streams lie on a common
// axis and can be compared directly.
int RunTimeline(Pipeline* p, int streams, int repeat) {
  double submit_ms = 0.0;
  double total_ms = 0.0;
  for (int i = 0; i < kWarmupRuns; ++i) {
    ACL_CHECK(RunPipeline(p, streams, repeat, &submit_ms, &total_ms));
  }
  ACL_CHECK(RunPipeline(p, streams, repeat, &submit_ms, &total_ms));
  std::printf(
      "[PERF] tag=timeline chunks=%d streams=%d repeat=%d "
      "submit_ms=%.4f total_ms=%.4f\n",
      p->chunks, streams, repeat, submit_ms, total_ms);

  for (int c = 0; c < p->chunks; ++c) {
    uint64_t ts[kEventsPerChunk] = {0, 0, 0, 0};
    for (int j = 0; j < kEventsPerChunk; ++j) {
      ACL_CHECK(aclrtEventGetTimestamp(p->event[c][j], &ts[j]));
    }
    std::printf(
        "[TL] chunk=%d stream=%d t_begin=%llu t_h2d=%llu t_comp=%llu "
        "t_d2h=%llu\n",
        c, c % streams, static_cast<unsigned long long>(ts[0]),
        static_cast<unsigned long long>(ts[1]),
        static_cast<unsigned long long>(ts[2]),
        static_cast<unsigned long long>(ts[3]));
  }
  return ACL_SUCCESS;
}


// Compares two ways of expressing the same cross-stream dependency. Both do
// the same work; only the party that waits differs.
int RunDependency(Pipeline* p, int repeat, int runs) {
  if (p->chunks < 3) {
    std::fprintf(stderr, "[ERR] api=RunDependency code=- msg=need 3 chunks\n");
    return ACL_ERROR_INVALID_PARAM;
  }
  aclrtEvent gate = nullptr;
  ACL_CHECK(aclrtCreateEventExWithFlag(&gate, kSyncFlag));
  const int long_chain = repeat * 4;

  // The two variants are interleaved run by run instead of being measured one
  // after the other. Under a fixed order whichever variant runs first absorbs
  // the process-level one-off cost, which is exactly the bias described in
  // section 13; interleaving spreads that cost over both variants.
  double total_sum[2] = {0.0, 0.0};
  double blocked_sum[2] = {0.0, 0.0};
  for (int i = 0; i < kWarmupRuns + runs; ++i) {
    for (int variant = 0; variant < 2; ++variant) {
      const bool use_event = (variant == 1);
      const double t0 = GetTimeMs();
      // A long chain of work on stream 0.
      for (int r = 0; r < long_chain; ++r) {
        ACL_CHECK(SubmitAdd(p->in_tensor[0], p->bias_tensor, p->alpha,
                            p->out_tensor[0], p->workspace[0],
                            p->workspace_size, p->stream[0]));
      }
      if (use_event) {
        ACL_CHECK(aclrtRecordEvent(gate, p->stream[0]));
        // Work that does not depend on stream 0 is submitted before the wait,
        // so it can start immediately instead of queueing behind the gate.
        for (int r = 0; r < repeat; ++r) {
          ACL_CHECK(SubmitAdd(p->in_tensor[1], p->bias_tensor, p->alpha,
                              p->out_tensor[1], p->workspace[1],
                              p->workspace_size, p->stream[1]));
        }
        ACL_CHECK(aclrtStreamWaitEvent(p->stream[1], gate));
      } else {
        // The host itself waits, so nothing can be submitted meanwhile.
        ACL_CHECK(aclrtSynchronizeStream(p->stream[0]));
        for (int r = 0; r < repeat; ++r) {
          ACL_CHECK(SubmitAdd(p->in_tensor[1], p->bias_tensor, p->alpha,
                              p->out_tensor[1], p->workspace[1],
                              p->workspace_size, p->stream[1]));
        }
      }
      // Work that does depend on stream 0: it reads what stream 0 wrote.
      // The destination is a third chunk so that no operator writes to one of
      // its own inputs, which aclnnAdd does not promise to support.
      for (int r = 0; r < repeat; ++r) {
        ACL_CHECK(SubmitAdd(p->out_tensor[0], p->bias_tensor, p->alpha,
                            p->out_tensor[2], p->workspace[1],
                            p->workspace_size, p->stream[1]));
      }
      const double t_submitted = GetTimeMs();
      ACL_CHECK(aclrtSynchronizeStream(p->stream[0]));
      ACL_CHECK(aclrtSynchronizeStream(p->stream[1]));
      if (i >= kWarmupRuns) {
        total_sum[variant] += GetTimeMs() - t0;
        blocked_sum[variant] += t_submitted - t0;
      }
    }
  }
  for (int variant = 0; variant < 2; ++variant) {
    std::printf("[DEPEND] variant=%s total_ms=%.4f submit_phase_ms=%.4f\n",
                (variant == 1) ? "event" : "stream_sync",
                total_sum[variant] / runs, blocked_sum[variant] / runs);
  }
  ACL_CHECK(aclrtDestroyEvent(gate));
  return ACL_SUCCESS;
}



### 8.6 模式分发与主程序

五种模式共用同一个 `Pipeline`。`timing` 与 `depend` 只用到第 0、1 块，其余模式用到全部分块。


In [ ]:
%%writefile -a src_event/acl_event.cpp

int RunMode(const char* mode, Pipeline* p, int streams, int repeat) {
  if (std::strcmp(mode, "timing") == 0) {
    ACL_CHECK(RunThreeClocks(p, repeat, kMeasureRuns));
    return ACL_SUCCESS;
  }
  if (std::strcmp(mode, "timeline") == 0) {
    ACL_CHECK(CreateMarkers(p, kTimingFlag));
    ACL_CHECK(RunTimeline(p, streams, repeat));
    return VerifyResult(p, "timeline");
  }
  if (std::strcmp(mode, "depend") == 0) {
    return RunDependency(p, repeat, kMeasureRuns);
  }
  std::fprintf(stderr, "[ERR] api=main code=- msg=unknown mode %s\n", mode);
  return ACL_ERROR_INVALID_PARAM;
}

int main(int argc, char** argv) {
  const char* mode = (argc > 1) ? argv[1] : "timing";
  const int total_mb = (argc > 2) ? std::atoi(argv[2]) : 64;
  const int chunks = (argc > 3) ? std::atoi(argv[3]) : 8;
  const int streams = (argc > 4) ? std::atoi(argv[4]) : 2;
  const int repeat = (argc > 5) ? std::atoi(argv[5]) : 4;
  const int64_t total_elems = static_cast<int64_t>(total_mb) * 1024 * 1024 / 4;

  // Validate the command line before allocating anything: the arrays below are
  // fixed size, only kMaxStreams streams are ever created, and the pipeline
  // assumes every chunk has exactly the same length.
  if (total_mb <= 0 || chunks <= 0 || chunks > kMaxChunks || streams <= 0 ||
      streams > kMaxStreams || repeat <= 0 || total_elems % chunks != 0) {
    std::fprintf(stderr,
                 "[ERR] api=main code=- msg=usage: acl_event <mode> <total_mb> "
                 "<chunks(1-%d)> <streams(1-%d)> <repeat>, and total_mb*262144 "
                 "must be divisible by chunks\n",
                 kMaxChunks, kMaxStreams);
    return ACL_ERROR_INVALID_PARAM;
  }

  std::printf("[INFO] acl_event start mode=%s\n", mode);
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(kDeviceId));
  aclrtContext context = nullptr;
  ACL_CHECK(aclrtCreateContext(&context, kDeviceId));

  Pipeline pipeline = {};
  ACL_CHECK(AllocPipeline(&pipeline, total_elems, chunks, kMaxStreams));
  std::printf(
      "[CFG] total_mb=%d chunks=%d streams=%d repeat=%d chunk_kb=%lld\n",
      total_mb, chunks, streams, repeat,
      static_cast<long long>(pipeline.chunk_elems * 4 / 1024));

  const int ret = RunMode(mode, &pipeline, streams, repeat);

  ACL_CHECK(FreePipeline(&pipeline));
  ACL_CHECK(aclrtDestroyContext(context));
  ACL_CHECK(aclrtResetDevice(kDeviceId));
  ACL_CHECK(aclFinalize());

  std::printf("[RESULT] %s\n", (ret == ACL_SUCCESS) ? "PASS" : "FAILED");
  std::printf("[INFO] acl_event finished\n");
  return (ret == ACL_SUCCESS) ? 0 : ret;
}


## 9. 编译与运行

链接三个库：Runtime 库、算子公共数据类型库与 Math 类算子库。


In [ ]:
import os, subprocess

SRC = "src_event/acl_event.cpp"
EXE = "src_event/acl_event"

# g++ [源文件] -I[头文件目录] [库目录] [库名] -o [可执行文件]
cmd = (
    ["g++", SRC, "-std=c++17", "-O2", "-Wall"]
    + ["-I" + os.environ["ASCEND_HOME_PATH"] + "/include"]
    + ACL_LIBDIRS
    + ACL_LIBS
    + ["-o", EXE]
)
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)


三种模式各运行一次。参数取 §7.1 定下的值：总量 64 MB、8 块、2 条 Stream、计算重复 4 次。


In [ ]:
import subprocess

TOTAL_MB, CHUNKS, STREAMS, REPEAT = 64, 8, 2, 4
MODES = ["timing", "timeline", "depend"]
out = {}

for mode in MODES:
    args = [str(TOTAL_MB), str(CHUNKS), str(STREAMS), str(REPEAT)]
    proc = subprocess.run(
        ["./" + EXE, mode] + args, capture_output=True, text=True, timeout=1800
    )
    out[mode] = proc.stdout
    flag = "✅" if proc.returncode == 0 else "❌ 返回码 %d" % proc.returncode
    print("%-9s %s  输出 %2d 行" % (mode, flag, len(proc.stdout.splitlines())))
    if proc.returncode != 0:
        print(proc.stderr)

print()
print(out["timing"])


## 10. 解析输出

三类记录行分别对应三组测量：`[CLOCK]` 三种计时方式、`[TL]` 时间线、`[DEPEND]` 依赖对照。

`[TL]` 行的四个时间戳是**绝对时刻，单位为微秒**，基准是设备启动以来的时间。它们的绝对值没有意义（数量级在 $10^9$），有意义的是它们之间的先后与间隔，因此下面先减去最小值把零点移到本次运行的起点。


In [ ]:
NUMERIC = {
    "chunk",
    "stream",
    "chunks",
    "streams",
    "repeat",
    "t_begin",
    "t_h2d",
    "t_comp",
    "t_d2h",
    "submit_ms",
    "wall_ms",
    "device_ms",
    "overhead_ms",
    "total_ms",
    "submit_phase_ms",
}


def parse(text, tag):
    # 把形如 [TAG] k=v k=v 的记录行解析为 dict 列表
    rows = []
    for line in text.splitlines():
        if not line.startswith(tag + " "):
            continue
        row = {}
        for item in line.split()[1:]:
            key, value = item.split("=", 1)
            row[key] = float(value) if key in NUMERIC else value
        rows.append(row)
    return rows


clock = parse(out["timing"], "[CLOCK]")[0]
tl = parse(out["timeline"], "[TL]")
depend = parse(out["depend"], "[DEPEND]")

# 把绝对时间戳换算成以本次运行起点为零、单位为毫秒的相对时刻
T0 = min(r["t_begin"] for r in tl)
for r in tl:
    for key in ("t_begin", "t_h2d", "t_comp", "t_d2h"):
        r[key + "_ms"] = (r[key] - T0) / 1000.0

print("[CLOCK] 1 条、[TL] %d 条、[DEPEND] %d 条" % (len(tl), len(depend)))
print()
print(
    "时间线的跨度：%.3f ms（%d 块 × %d 流）"
    % (max(r["t_d2h_ms"] for r in tl), CHUNKS, STREAMS)
)


## 11. 三种计时方式的对比

三个数字测的是同一段工作，差别只在覆盖范围。这段工作是：把 64 MB 一次性搬到设备、在**第 0 块**上重复 R 次算子、再把 64 MB 搬回——**与 §12 的分块流水线不是同一个工作量**，两节的绝对耗时不能直接相比。

**请先看 `submit_ms` 这一行**——它是下发这一整串任务所花的时间，与另外两个数相差约两个数量级。


In [ ]:
print("%-22s %12s %s" % ("计时方式", "耗时 (ms)", "覆盖的过程"))
print("-" * 72)
rows = [
    ("主机侧 · 不同步", clock["submit_ms"], "只有下发"),
    ("主机侧 · 同步后", clock["wall_ms"], "下发 + 设备执行 + 完成通知 + 主机唤醒"),
    ("设备侧 · Event", clock["device_ms"], "设备上任务的净执行区间"),
]
for name, value, cover in rows:
    print("%-22s %12.4f %s" % (name, value, cover))
print()
print(
    "主机侧开销 = 同步后 − 设备侧 = %.4f ms（占端到端 %.1f%%）"
    % (clock["overhead_ms"], 100 * clock["overhead_ms"] / clock["wall_ms"])
)
print("下发耗时占端到端 %.2f%%" % (100 * clock["submit_ms"] / clock["wall_ms"]))
print()
print(
    "若把下发耗时当作这段工作的耗时来汇报，会低估 %.0f 倍。"
    % (clock["wall_ms"] / max(clock["submit_ms"], 1e-9))
)


## 12. 时间线：流水线中哪些阶段真正重叠

这是本实验的核心一节。§4.1 的两张示意图画的是任务之间的先后关系，下面这张甘特图把同一件事画在时间轴上——**每一个色块的两个边界都是一个 `aclrtEventGetTimestamp` 读回的时间戳**。它把每一块数据的三个阶段画在它所属的 Stream 那一行上，横轴是时间。**由于全部 Event 共用设备上的同一个时钟，两条 Stream 的色块可以直接比较先后。**

读图之前先明确一件事：**色块的边界是时刻，宽度是两个时刻之差。** `aclrtRecordEvent` 捕获的是调用时该 Stream 上**已下发的全部任务**（§2.4），因此一个标记的时间戳，是它前面那些任务全部完成的时刻。由此得到两条读图规则：

- **判断两段是否在时间上重叠要看边界，结论可靠**，因为边界都是实测得到的时刻；
- **判断某一段的快慢要看宽度，但需留有余地**，因为一段的宽度中可能含有排队等待的时间。以每块的第一段（H2D）为例，它的左边界是该 Stream 上此前任务全部完成的时刻（对第一块而言就是本次运行的起点），右边界是本块 H2D 完成的时刻；若这段时间里传输链路正被另一条 Stream 占用，等待也会算进这一段。

按顺序看三件事：

1. **同一条 Stream 内**，每一块的三段是否首尾相接？应当是——同一 Stream 内的任务严格保序。
2. **两条 Stream 之间**，计算段与传输段有没有在时间上并排出现？若有，说明计算与传输确实重叠。
3. **两条 Stream 之间，H2D 色块与 D2H 色块有没有并排出现？** 这两个方向走的是同一条主机与设备之间的链路，能否真的同时推进取决于本机硬件。注意两条 Stream 的相位难免错开，**边界处出现少量交叠是常态**，要看的是有没有成片的并排。

还有一处容易看漏：**两条 Stream 未必同时起步。** 若图上一行的第一块明显晚于另一行开始，说明主机把任务交给第二条 Stream 时，第一条上的搬运已经开始了。这也是设备侧的真实观测，它会让整条时间线的跨度大于任何单条 Stream 自己的跨度。


In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

C_H2D, C_COMP, C_D2H = "#3B6FE0", "#E07A3B", "#2E9E6B"

fig, ax = plt.subplots(figsize=(12.0, 2.4 + 0.5 * STREAMS), dpi=120)
for r in tl:
    y = r["stream"]
    for start, end, color in (
        (r["t_begin_ms"], r["t_h2d_ms"], C_H2D),
        (r["t_h2d_ms"], r["t_comp_ms"], C_COMP),
        (r["t_comp_ms"], r["t_d2h_ms"], C_D2H),
    ):
        ax.barh(
            y,
            end - start,
            left=start,
            height=0.55,
            color=color,
            edgecolor="white",
            lw=0.4,
        )
    ax.text(
        (r["t_begin_ms"] + r["t_d2h_ms"]) / 2,
        y + 0.42,
        "c%d" % int(r["chunk"]),
        ha="center",
        va="bottom",
        fontsize=8,
        color="#444444",
    )

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (C_H2D, C_COMP, C_D2H)]
ax.legend(handles, ["H2D", "Compute", "D2H"], frameon=False, ncol=3, loc="upper center")
ax.set_yticks(range(STREAMS))
ax.set_yticklabels(["stream %d" % s for s in range(STREAMS)])
ax.set_xlabel("Device time since start of run (ms)")
ax.set_title(
    "Lab 4: device-side timeline (%d chunks over %d streams)" % (CHUNKS, STREAMS)
)
ax.set_ylim(-0.7, STREAMS - 0.1)
ax.grid(axis="x", alpha=0.3)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()


## 13. 跨流依赖：Event 与流同步的对照

两个版本做完全相同的工作：stream0 上一条长任务链，stream1 上一段**不依赖**它的任务和一段**依赖**它的任务。差别只在依赖的表达方式。

- `stream_sync`：主机调用 `aclrtSynchronizeStream(stream0)` 等它做完，再向 stream1 下发全部任务。**主机被挡住的这段时间里，那段本可以先跑的独立任务连下发的机会都没有。**
- `event`：`aclrtRecordEvent(gate, stream0)` 之后**先下发独立任务**，再 `aclrtStreamWaitEvent(stream1, gate)`，最后下发依赖任务。主机全程不阻塞，独立任务立即开始执行。

`submit_phase_ms` 是主机从开始到把全部任务下发完所用的时间，它直接反映主机被占用了多久。

> 本节的两个版本是**逐轮交错**执行的：每一轮依次跑一次 `stream_sync` 和一次 `event`，重复若干轮后再分别求平均。若按固定顺序先跑完一个版本再跑另一个，序列中第一个要承担进程级的一次性开销，两版之差里就会混进一份顺序红利。交错执行把这份开销摊到两个版本上，代价是每一轮都要在两种写法之间切换。


In [ ]:
NAME = {"stream_sync": "流同步（主机等）", "event": "Event（设备等）"}
print("%-20s %12s %16s" % ("依赖表达方式", "总耗时ms", "主机下发阶段ms"))
print("-" * 52)
for r in depend:
    print(
        "%-20s %12.4f %16.4f"
        % (NAME[r["variant"]], r["total_ms"], r["submit_phase_ms"])
    )

a = next(r for r in depend if r["variant"] == "stream_sync")
b = next(r for r in depend if r["variant"] == "event")
print()
print(
    "总耗时之差       = %+.4f ms（Event 版相对流同步版）"
    % (b["total_ms"] - a["total_ms"])
)
print("主机占用之差     = %+.4f ms" % (b["submit_phase_ms"] - a["submit_phase_ms"]))
print()
cut = 1.0 - b["submit_phase_ms"] / a["submit_phase_ms"]
print("主机下发阶段缩短了 %.0f%%。" % (100 * cut))
if cut > 0.1:
    print("→ Event 版的主机没有被挡在同步调用里，因而更早把全部任务交给了设备。")
    print("  缩短的幅度取决于 stream0 任务链的长度：链越长，主机被挡住的时间越长。")
else:
    print("→ 两版的主机下发阶段接近。请检查 stream0 的任务链是否足够长——")
    print("  任务太短时主机还没下发完设备就已算完，两种写法看不出差别。")


## 14. 结果分析

> 本节只陈述关系与判据——所有数据请以本机 §11–§13 的输出为准。

**① 三种计时方式的差距不是精度问题，而是覆盖范围问题。**

§11 的三行测的是同一段工作。不同步的那一行只覆盖下发，通常只有端到端的**百分之一量级**；把它当作这段工作的耗时来汇报，会低估一到两个数量级。它并不是测不准，只是测的是下发而不是执行。

端到端减设备侧得到的**主机侧开销**，包含下发、完成通知回传与主机线程唤醒。它占端到端的比例随机器而异，请读本机打印的那个百分比。由此得到一条通用判据：**一个性能数字是否可信，先问它覆盖了哪一段过程。**

**② 甘特图把「哪两段重叠」从推断变成了观测。**

请对照 §12 的图回答两件事：计算段与传输段有没有并排出现？H2D 色块与 D2H 色块有没有成片并排？前者用的是 AI Core 与 DMA 两类不同的执行单元；后者共用同一条主机与设备之间的链路，**能否同时推进取决于本机硬件**，请用本机的图判断。

**③ Event 相对流同步的收益，体现在主机被占用的时间上。**

§13 的两个版本工作量完全相同，差别只在依赖的表达方式。Event 版把不依赖 stream0 的那段任务放在 `aclrtStreamWaitEvent` **之前**下发，因此主机不必停下等待；流同步版则被挡在同步调用里，这段任务连下发的机会都没有。请读两行 `submit_phase_ms` 之差。

**没有独立任务可以先下发的场景测不出这个差别。** Event 的价值不在于缩短设备端的执行时间，而在于把等待从主机线程转移到设备侧。

---

### 🎓 结论

**Event 时间戳测量的是设备侧的时间。** 主机侧墙钟测到的是从下发到同步返回的整个区间，其中包含下发、完成通知与主机线程唤醒；Event 时间戳测到的是设备上任务的执行区间，而 `aclrtEventGetTimestamp` 给出的绝对时刻还可以把多条 Stream 放到同一根时间轴上比较。

**Event 也是跨流依赖的表达方式。** `aclrtRecordEvent` 在一条流上打标记、`aclrtStreamWaitEvent` 让另一条流等这个标记，等待发生在设备侧，主机线程不被阻塞。

两者指向同一件事：**在异步的执行模型中，任务何时完成必须被显式地表达和测量，它不会自动变得可观测。**


## 15. 🔧 动手练习

> **提示**：本题要改 C++ 源码。修改源码后需要重新执行 `%%writefile` 单元格；由于 §8.1 为覆盖写、其后五个为追加写，**须从 8.1 开始按顺序重新执行**。

**把 flag 写错，观察三种不同的失败方式。**

本题只需三次改动，每次改一处、跑一次、记下现象：

1. 把创建计时用 Event 的 flag 由 `ACL_EVENT_TIME_LINE` 改为 `ACL_EVENT_SYNC`，重跑 `timeline` 模式。`aclrtEventGetTimestamp` 会怎样？返回码是多少？
2. 改回 `ACL_EVENT_TIME_LINE`，但把 `aclrtEventElapsedTime` 的两个 Event 参数**对调**，观察算出的时间差。
3. 在任何一个 Event 上，把 `aclrtRecordEvent` 整行注释掉，再对它调用 `aclrtEventElapsedTime`。

三次的失败方式各不相同：一次是能力不匹配、一次是参数顺序、一次是时序前提未满足。请分别写下现象、返回码与你据此得到的排查线索。**这三类错误在写 Event 计时代码时都很常见，且都不会在编译期暴露。**


## 16. 🤔 思考题

1. §11 显示不同步的那一种远小于同步之后的那一种。设想以下情形：某人用不同步的方式测出 0.01 ms 的算子耗时并写进报告。请说明这个数字实际测的是什么，以及应当怎样描述才不会误导读者。

2. §3.1 第二条前提要求起止两个 Event 与被测任务在同一条 Stream 上。若把起始 Event 记在 stream0、结束 Event 记在 stream1，`aclrtEventElapsedTime` 算出的差值还有意义吗？请说明理由。


## 17. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| Event 的四种用途 | 跨流依赖、设备侧计时、非阻塞查询、主机等待；**同一个对象，flag 不同** |
| `ACL_EVENT_SYNC` | 使能多 Stream 同步；**缺少它则不能用于 `aclrtStreamWaitEvent`** |
| `ACL_EVENT_TIME_LINE` | 使能时间戳；按接口说明，使能后会影响 Event 相关接口的性能 |
| `ACL_EVENT_CAPTURE_STREAM_PROGRESS` | 捕获 Stream 执行进度；**`aclrtQueryEventStatus` 与 `aclrtSynchronizeEvent` 用的是它** |
| 创建接口 | 同步、计时、查询三类用途一律用 **`aclrtCreateEventExWithFlag`**；`aclrtCreateEventWithFlag` 只用于 Graph 捕获场景的 external Event |
| Event 数量 | 上限以本机 `aclrtGetEventAvailNum` 的实测值为准；**资源在 `aclrtRecordEvent` 时才申请** |
| Record 的语义 | 捕获**调用时已下发的全部任务**；同一 Event 可多次 Record，每次覆盖 |
| 两个调用的顺序 | `aclrtRecordEvent(event, stream)` 与 `aclrtStreamWaitEvent(stream, event)` **参数顺序相反**；**下发时必须先 Record 再 Wait**，颠倒不报错，只会偶发地读到未完成的数据 |
| 设备侧计时四条前提 | 带 `TIME_LINE` 创建；起止与任务同流；**先同步再取值**；输出为 `float` 毫秒 |
| 绝对时间戳 | `aclrtEventGetTimestamp` 返回微秒级绝对时刻，**是跨 Stream 画时间线的唯一依据** |
| 三种计时方式 | 不同步测下发、同步后测端到端、Event 测设备侧执行区间；**差别是覆盖范围而非精度** |
| Event 相对流同步 | 阻塞的是**设备侧的目标 Stream** 而非主机线程；价值在有独立任务可先下发时显现 |
| 非阻塞查询与内存复用 | `aclrtQueryEventStatus` 不阻塞；在最后一次用到某块内存之后 Record，Event 变 `COMPLETE` 即可复用，**不必等整条 Stream** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Event 的四种用途</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">跨流依赖、设备侧计时、非阻塞查询、主机等待；<strong>同一个对象，flag 不同</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_SYNC</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">使能多 Stream 同步；<strong>缺少它则不能用于 <code>aclrtStreamWaitEvent</code></strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_TIME_LINE</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">使能时间戳；按接口说明，使能后会影响 Event 相关接口的性能</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_EVENT_CAPTURE_STREAM_PROGRESS</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">捕获 Stream 执行进度；<strong><code>aclrtQueryEventStatus</code> 与 <code>aclrtSynchronizeEvent</code> 用的是它</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">创建接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同步、计时、查询三类用途一律用 <strong><code>aclrtCreateEventExWithFlag</code></strong>；<code>aclrtCreateEventWithFlag</code> 只用于 Graph 捕获场景的 external Event</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Event 数量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">上限以本机 <code>aclrtGetEventAvailNum</code> 的实测值为准；<strong>资源在 <code>aclrtRecordEvent</code> 时才申请</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Record 的语义</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">捕获<strong>调用时已下发的全部任务</strong>；同一 Event 可多次 Record，每次覆盖</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两个调用的顺序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtRecordEvent(event, stream)</code> 与 <code>aclrtStreamWaitEvent(stream, event)</code> <strong>参数顺序相反</strong>；<strong>下发时必须先 Record 再 Wait</strong>，颠倒不报错，只会偶发地读到未完成的数据</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">设备侧计时四条前提</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">带 <code>TIME_LINE</code> 创建；起止与任务同流；<strong>先同步再取值</strong>；输出为 <code>float</code> 毫秒</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">绝对时间戳</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtEventGetTimestamp</code> 返回微秒级绝对时刻，<strong>是跨 Stream 画时间线的唯一依据</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三种计时方式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不同步测下发、同步后测端到端、Event 测设备侧执行区间；<strong>差别是覆盖范围而非精度</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Event 相对流同步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">阻塞的是<strong>设备侧的目标 Stream</strong> 而非主机线程；价值在有独立任务可先下发时显现</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">非阻塞查询与内存复用</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtQueryEventStatus</code> 不阻塞；在最后一次用到某块内存之后 Record，Event 变 <code>COMPLETE</code> 即可复用，<strong>不必等整条 Stream</strong></td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **异构应用的性能问题，首先是数据搬运问题，其次才是计算问题。**

本实验没有改进任何搬运，它做的是**把搬运测清楚**：能重叠多少取决于硬件允许哪两类任务并发，这既不能由代码推出，也不能只靠端到端耗时反推，需要合适的手段去观测。
